In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

In [ ]:
import zipfile
import os

DATA_DIR = 'f1/'
if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile('archive.zip', 'r') as z:
        z.extractall('f1')

In [ ]:
def load(name):
    return pd.read_csv(f'{DATA_DIR}{name}.csv', na_values=['\\N'])

results      = load('results')
races        = load('races')
drivers      = load('drivers')
constructors = load('constructors')
circuits     = load('circuits')
status       = load('status')
qualifying   = load('qualifying')

for n, d in {'results': results, 'races': races, 'drivers': drivers,
             'constructors': constructors, 'circuits': circuits,
             'status': status, 'qualifying': qualifying}.items():
    print(f'{n:13s}', d.shape)

In [ ]:
results = results.rename(columns={'position': 'finish_position',
                                  'number': 'car_number'})

races = races[['raceId', 'year', 'round', 'circuitId', 'name', 'date']] \
        .rename(columns={'name': 'race_name'})

circuits = circuits[['circuitId', 'name', 'country', 'lat', 'lng', 'alt']] \
        .rename(columns={'name': 'circuit_name', 'country': 'circuit_country'})

drivers = drivers[['driverId', 'forename', 'surname', 'dob', 'nationality']] \
        .rename(columns={'nationality': 'driver_nationality'})

constructors = constructors[['constructorId', 'name', 'nationality']] \
        .rename(columns={'name': 'constructor_name',
                         'nationality': 'constructor_nationality'})

qualifying = qualifying[['raceId', 'driverId', 'position']] \
        .rename(columns={'position': 'quali_position'})

In [ ]:
n_before = len(results)
df = (results
      .merge(races,        on='raceId',        how='left', validate='many_to_one')
      .merge(circuits,     on='circuitId',     how='left', validate='many_to_one')
      .merge(drivers,      on='driverId',      how='left', validate='many_to_one')
      .merge(constructors, on='constructorId', how='left', validate='many_to_one')
      .merge(status,       on='statusId',      how='left', validate='many_to_one')
      .merge(qualifying,   on=['raceId', 'driverId'], how='left', validate='many_to_one'))

print('Lignes avant/après merge :', n_before, len(df))
assert n_before == len(df)
print(df.shape)
df.head()

In [ ]:
df = df[df['year'] >= 2003].copy()

df['date'] = pd.to_datetime(df['date'])
df['dob']  = pd.to_datetime(df['dob'])

df['driver'] = df['forename'] + ' ' + df['surname']
df['driver_age'] = ((df['date'] - df['dob']).dt.days / 365.25).round(1)

def lap_to_seconds(t):
    if pd.isna(t):
        return np.nan
    try:
        parts = str(t).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        elif len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except Exception:
        pass
    return np.nan

df['fastest_lap_sec'] = df['fastestLapTime'].apply(lap_to_seconds)

print(len(df))

In [ ]:
df['podium'] = (df['positionOrder'] <= 3).astype(int)
df['win']    = (df['positionOrder'] == 1).astype(int)
df['finished'] = (df['status'] == 'Finished') | df['status'].fillna('').str.startswith('+')
df['DNF'] = (~df['finished']).astype(int)

In [ ]:
df.loc[df['grid'] == 0, 'grid'] = df['grid'].max() + 1

drop_cols = ['resultId', 'raceId', 'driverId', 'constructorId', 'circuitId', 'statusId',
             'forename', 'surname', 'dob', 'fastestLapTime', 'positionText',
             'time', 'milliseconds', 'fastestLapSpeed', 'car_number', 'finished']
df = df.drop(columns=drop_cols, errors='ignore')

before = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {before - len(df)}')

missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

df.to_csv('f1_cleaned.csv', index=False)
print('Cleaned shape :', df.shape)